# ᚱ Viking Rune Stones: High-Accuracy Pre-Training Engine (YOLOv8)
### Advanced 3D Epigraphic Simulation & Curriculum Pre-Training Pipeline
---

### 🌟 ما الجديد في هذا الدفتر لرفع الدقة بشكل استثنائي؟
1. **دقة إدخال فائقة (`imgsz=1024`):** مضاعفة دقة الإدخال للحفاظ على الخطوط الرونية الدقيقة ومنع انضغاطها عند التصغير.
2. **محاكاة نقش إزميل ثلاثي الأبعاد (3D Raking-Light Chisel Physics):**
   - حفر ثنائي اللون (Shadow + Highlight) يحاكي الإضاءة المائلة للشمس على الأحجار الحقيقية.
   - حواف متآكلة (Weathered Micro-Chipping) وانحناءات طبيعية في خطوط الحفر باليد.
3. **توليد مشتتات صخرية سلبية (Negative Distractors):**
   - تشققات صخرية طبيعية، عروق كوارتز، ومؤثرات تآكل غير معلّمة لتعليم YOLO عدم الخلط بين شقوق الصخور والحروف.
4. **معمارية متطورة (`yolov8m.pt`):**
   - قدرة تمثيلية أعلى بـ 25 مليون بارامتر للتفريق الدقيق بين الحروف الـ 17 المتشابهة.
5. **تحكم دقيق في التدريب (Epigraphic Training Control):**
   - الحفاظ التام على اتجاه الحروف (`fliplr=0.0`).
   - إيقاف تشويه الموازييك في المراحل الحاسمة (`close_mosaic=15`).
   - تدرج جيبي سلس لمعدل التعلم (`cos_lr=True`).
6. **دعم التغذية الهجينة والتقييم على الأحجار الحقيقية:**
   - فحص وتضمين صور حقيقية تلقائياً إن وُجدت، وتقييم فوري شامل.


## 1. تجهيز بيئة العمل وتسريع كرت الشاشة (GPU Setup)
التأكد من توفر معالج الرسومات (NVIDIA T4 / V100 / A100) وتثبيت أحدث حزم Ultralytics و OpenCV.


In [ ]:
# التحقق من كرت الشاشة
!nvidia-smi

# تثبيت أحدث حزم YOLO ومعالجة الصور
!pip install -q "ultralytics>=8.2.0" opencv-python pillow matplotlib seaborn pyyaml

import os
import sys
import shutil
import random
import math
import zipfile
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import torch
import yaml

print(f"\n✓ PyTorch Version : {torch.__version__}")
print(f"✓ CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ Active GPU      : {torch.cuda.get_device_name(0)}")
    print(f"✓ Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ تحذير: كرت الشاشة غير مفعل! توجه إلى: Runtime -> Change runtime type -> اختر T4 GPU.")


## 2. ربط Google Drive (لحفظ الأوزان ومنع فقدان البيانات)
ربط الحساب لضمان حفظ كل نقطة تفتيش (Checkpoints) والموديل الخبير النهائي تلقائياً في حسابك.


In [ ]:
USE_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/Viking_Rune_HighAccuracy")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
    print(f"✓ تم ربط Google Drive بنجاح! مجلد الحفظ:
👉 {DRIVE_PROJECT}")
except Exception as e:
    print("⚠️ تعذر ربط Google Drive، سيتم الحفظ محلياً داخل بيئة Colab المؤقتة.")
    USE_DRIVE = False
    DRIVE_PROJECT = Path("/content/viking_project")
    DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)


## 3. محرك المحاكاة الأثرية المتقدم (Advanced 3D Chisel & Rock Engine)
توليد نقوش فايكنج دقيقة طبقاً للـ Younger Futhark الأصلي مع خامات صخرية ثلاثية الأبعاد وظلال حفر حقيقية وتشققات سلبية.


In [ ]:
# ==============================================================================
# الأبجدية الرونية الاسكندنافية (Younger Futhark - 16 رون أصلي + فاصل الكلمات)
# ==============================================================================
RUNES = [
    {
        "id": 0, "name": "fehu", "latin": "F",  # ᚠ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.45), (0.52, 0.35), (0.75, 0.20)], [(0.35, 0.70), (0.52, 0.60), (0.75, 0.45)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.42), (0.75, 0.18)], [(0.35, 0.68), (0.75, 0.42)]]
        ]
    },
    {
        "id": 1, "name": "uruz", "latin": "U",  # ᚢ
        "variants": [
            [[(0.30, 0.32), (0.30, 0.90)], [(0.30, 0.32), (0.34, 0.18), (0.50, 0.12), (0.66, 0.18), (0.70, 0.32)], [(0.70, 0.32), (0.70, 0.90)]],
            [[(0.28, 0.30), (0.28, 0.92)], [(0.28, 0.30), (0.50, 0.10), (0.72, 0.30)], [(0.72, 0.30), (0.72, 0.92)]]
        ]
    },
    {
        "id": 2, "name": "thurisaz", "latin": "Th",  # ᚦ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.28), (0.55, 0.28), (0.72, 0.38), (0.72, 0.58), (0.55, 0.68), (0.35, 0.68)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.25), (0.75, 0.48), (0.35, 0.72)]]
        ]
    },
    {
        "id": 3, "name": "ansuz", "latin": "A",  # ᚬ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.28), (0.75, 0.40)], [(0.25, 0.48), (0.75, 0.60)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.26), (0.78, 0.38)], [(0.22, 0.46), (0.78, 0.58)]]
        ]
    },
    {
        "id": 4, "name": "raidho", "latin": "R",  # ᚱ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.14), (0.55, 0.15), (0.70, 0.24), (0.70, 0.38), (0.55, 0.46), (0.35, 0.48)], [(0.35, 0.48), (0.70, 0.90)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.12), (0.72, 0.28), (0.35, 0.48)], [(0.35, 0.48), (0.72, 0.90)]]
        ]
    },
    {
        "id": 5, "name": "kaunan", "latin": "K",  # ᚴ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.50), (0.55, 0.38), (0.75, 0.20)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.48), (0.75, 0.18)]]
        ]
    },
    {
        "id": 6, "name": "hagalaz", "latin": "H",  # ᚼ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.35), (0.78, 0.65)], [(0.22, 0.65), (0.78, 0.35)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.38), (0.80, 0.62)], [(0.20, 0.62), (0.80, 0.38)]]
        ]
    },
    {
        "id": 7, "name": "naudiz", "latin": "N",  # ᚾ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.25, 0.38), (0.75, 0.62)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.28, 0.40), (0.72, 0.60)]]
        ]
    },
    {
        "id": 8, "name": "isaz", "latin": "I",  # ᛁ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)]],
            [[(0.48, 0.12), (0.52, 0.88)]]
        ]
    },
    {
        "id": 9, "name": "ar_jera", "latin": "A",  # ᛅ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.50, 0.50), (0.78, 0.70)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.50, 0.48), (0.76, 0.68)]]
        ]
    },
    {
        "id": 10, "name": "sowilo", "latin": "S",  # ᛋ
        "variants": [
            [[(0.65, 0.10), (0.35, 0.36)], [(0.35, 0.36), (0.65, 0.64)], [(0.65, 0.64), (0.35, 0.90)]],
            [[(0.62, 0.12), (0.38, 0.38)], [(0.38, 0.38), (0.62, 0.62)], [(0.62, 0.62), (0.38, 0.88)]]
        ]
    },
    {
        "id": 11, "name": "tiwaz", "latin": "T",  # ᛏ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.90)], [(0.20, 0.28), (0.50, 0.10)], [(0.50, 0.10), (0.80, 0.28)]],
            [[(0.50, 0.10), (0.50, 0.90)], [(0.22, 0.26), (0.50, 0.10), (0.78, 0.26)]]
        ]
    },
    {
        "id": 12, "name": "berkanan", "latin": "B",  # ᛒ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.12), (0.55, 0.12), (0.70, 0.24), (0.70, 0.38), (0.55, 0.48), (0.35, 0.48)], [(0.35, 0.48), (0.55, 0.48), (0.70, 0.60), (0.70, 0.76), (0.55, 0.88), (0.35, 0.88)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.12), (0.70, 0.30), (0.35, 0.48)], [(0.35, 0.48), (0.70, 0.68), (0.35, 0.88)]]
        ]
    },
    {
        "id": 13, "name": "mannaz", "latin": "M",  # ᛉ
        "variants": [
            [[(0.50, 0.30), (0.50, 0.90)], [(0.20, 0.10), (0.50, 0.30)], [(0.80, 0.10), (0.50, 0.30)]],
            [[(0.50, 0.32), (0.50, 0.90)], [(0.18, 0.12), (0.50, 0.32), (0.82, 0.12)]]
        ]
    },
    {
        "id": 14, "name": "laguz", "latin": "L",  # ᛚ
        "variants": [
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.10), (0.75, 0.35)]],
            [[(0.35, 0.10), (0.35, 0.90)], [(0.35, 0.10), (0.70, 0.38)]]
        ]
    },
    {
        "id": 15, "name": "yr", "latin": "R_final",  # ᛦ
        "variants": [
            [[(0.50, 0.10), (0.50, 0.70)], [(0.20, 0.90), (0.50, 0.70)], [(0.80, 0.90), (0.50, 0.70)]],
            [[(0.50, 0.10), (0.50, 0.68)], [(0.18, 0.88), (0.50, 0.68), (0.82, 0.88)]]
        ]
    }
]

SEPARATOR_DEF = {
    "id": 16,
    "name": "separator",
    "types": ["colon", "cross", "single_dot"]
}

CLASS_NAMES = [r["name"] for r in RUNES] + [SEPARATOR_DEF["name"]]

class BalancedRuneSampler:
    def __init__(self, runes):
        self.runes = runes
        self.n = len(runes)
        self.pool = []
        self._refill()

    def _refill(self):
        self.pool = list(range(self.n))
        random.shuffle(self.pool)

    def next_rune(self):
        if not self.pool:
            self._refill()
        return self.runes[self.pool.pop()]

# ------------------------------------------------------------------------------
# توليد خامات الصخور الطبيعية ثلاثية الأبعاد
# ------------------------------------------------------------------------------
def generate_photorealistic_rock(w=1024, h=1024, rock_type=None):
    if rock_type is None:
        rock_type = random.choice(["granite", "sandstone", "limestone", "weathered_slate"])

    if rock_type == "granite":
        base_rgb = np.array([150.0, 145.0, 140.0], dtype=np.float32)
        tint = np.array([random.uniform(-10, 15), random.uniform(-5, 5), random.uniform(-15, -2)])
    elif rock_type == "sandstone":
        base_rgb = np.array([175.0, 155.0, 125.0], dtype=np.float32)
        tint = np.array([random.uniform(5, 25), random.uniform(0, 12), random.uniform(-20, -5)])
    elif rock_type == "limestone":
        base_rgb = np.array([190.0, 185.0, 175.0], dtype=np.float32)
        tint = np.array([random.uniform(-5, 10), random.uniform(-5, 5), random.uniform(-10, 2)])
    else:  # weathered_slate
        base_rgb = np.array([85.0, 90.0, 95.0], dtype=np.float32)
        tint = np.array([random.uniform(-8, 5), random.uniform(-5, 5), random.uniform(0, 15)])

    # إضاءة شمسية مائلة
    gx = np.linspace(random.uniform(0.92, 1.0), random.uniform(1.0, 1.08), w)[None, :]
    gy = np.linspace(random.uniform(0.92, 1.0), random.uniform(1.0, 1.08), h)[:, None]
    lighting = gy * gx

    # نسيج الحبيبات الكبيرة
    c_grid = np.random.uniform(0.85, 1.15, (h // 25 + 1, w // 25 + 1)).astype(np.float32)
    c_map = np.array(
        Image.fromarray((c_grid * 128).astype(np.uint8)).resize((w, h), Image.BICUBIC),
        dtype=np.float32
    ) / 128.0

    # نسيج الحبيبات الرملية الدقيقة
    f_noise = np.random.normal(0, random.uniform(6.0, 12.0), (h, w, 1)).astype(np.float32)

    rock = (base_rgb + tint) * lighting[:, :, None] * c_map[:, :, None] + f_noise
    rock = np.clip(rock, 0, 255).astype(np.uint8)
    return Image.fromarray(rock, "RGB")

# ------------------------------------------------------------------------------
# إضافة تشققات وعروق صخرية سلبية (Negative Distractors)
# ------------------------------------------------------------------------------
def add_rock_cracks_and_veins(img):
    draw = ImageDraw.Draw(img)
    w, h = img.size
    num_cracks = random.randint(1, 4)
    for _ in range(num_cracks):
        cx, cy = random.randint(50, w - 50), random.randint(50, h - 50)
        pts = [(cx, cy)]
        length = random.randint(60, 240)
        angle = random.uniform(0, 2 * math.pi)
        for _ in range(length // 20):
            angle += random.uniform(-0.4, 0.4)
            step = random.uniform(15, 25)
            nx = pts[-1][0] + step * math.cos(angle)
            ny = pts[-1][1] + step * math.sin(angle)
            pts.append((nx, ny))
        # رسم شق حقيقي (ظل غامق + نور رفيع)
        crack_color = (random.randint(25, 45), random.randint(25, 45), random.randint(25, 45))
        draw.line(pts, fill=crack_color, width=random.choice([1, 2]))
    return img

# ------------------------------------------------------------------------------
# رسم حفر الإزميل ثلاثي الأبعاد (3D Beveled Chisel Stroke)
# ------------------------------------------------------------------------------
def draw_3d_chiseled_stroke(draw, pts, base_dark, stroke_w=4, light_angle=45):
    # تحويل الزاوية إلى إزاحة الظل والنور
    rad = math.radians(light_angle)
    off_x = int(round(1.5 * math.cos(rad)))
    off_y = int(round(1.5 * math.sin(rad)))

    highlight_col = (min(255, base_dark[0] + 130), min(255, base_dark[1] + 130), min(255, base_dark[2] + 125))
    shadow_col = (max(0, base_dark[0] - 25), max(0, base_dark[1] - 25), max(0, base_dark[2] - 25))

    # 1. رسم خط النور المنعكس على حافة الحفر
    hl_pts = [(x - off_x, y - off_y) for x, y in pts]
    draw.line(hl_pts, fill=highlight_col, width=max(1, stroke_w - 1))

    # 2. رسم عمق الحفر الرئيسي
    draw.line(pts, fill=shadow_col, width=stroke_w)

    # 3. رسم أعمق نقطة في الحفر (Core groove)
    sh_pts = [(x + off_x, y + off_y) for x, y in pts]
    core_dark = (max(0, shadow_col[0] - 15), max(0, shadow_col[1] - 15), max(0, shadow_col[2] - 15))
    draw.line(sh_pts, fill=core_dark, width=max(1, stroke_w - 2))

print("✓ تم تحميل محرك النقش الأثري المتطور بنجاح!")


## 4. المرحلة الأولى (Stage 1: تشريح السيقان والرموز الفردية)
- **الهدف:** تعويد الشبكة العصبية على أشكال الحروف الرونية الأساسية بدقة `1024×1024`.
- **الموديل:** `yolov8m.pt` (Medium) للتأسيس عالي الدقة.
- **البيانات:** 800 صورة تدريب + 150 صورة تحقق.


In [ ]:
d1 = 'dataset_stage1'
os.makedirs(f'{d1}/images/train', exist_ok=True); os.makedirs(f'{d1}/labels/train', exist_ok=True)
os.makedirs(f'{d1}/images/val', exist_ok=True); os.makedirs(f'{d1}/labels/val', exist_ok=True)

sampler_s1 = BalancedRuneSampler(RUNES)
counts_s1 = Counter()

def gen_s1_sample(idx, sampler, img_size=1024):
    img = generate_photorealistic_rock(img_size, img_size)
    img = add_rock_cracks_and_veins(img)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(40, 65), random.randint(40, 65), random.randint(40, 65))
    lw = random.randint(4, 6)

    num_rows = random.choice([1, 2])
    for r in range(num_rows):
        num_runes = random.randint(4, 7)
        band_w = num_runes * 110
        x_start = max(60, (img_size - band_w) // 2 + random.randint(-30, 30))
        y_top = 180 + r * 350 + random.randint(-20, 20)
        box_h = 240

        # خط تأطير علوي وسفلي
        if random.random() < 0.70:
            draw_3d_chiseled_stroke(draw, [(x_start - 30, y_top), (x_start + band_w + 30, y_top)], base_dark, stroke_w=3)
            draw_3d_chiseled_stroke(draw, [(x_start - 30, y_top + box_h), (x_start + band_w + 30, y_top + box_h)], base_dark, stroke_w=3)

        curr_x = x_start
        for _ in range(num_runes):
            rune = sampler.next_rune()
            counts_s1[rune["name"]] += 1
            variant = random.choice(rune["variants"])
            rw = random.randint(70, 95)
            tilt = random.uniform(-0.04, 0.04)

            min_x, min_y, max_x, max_y = 1e9, 1e9, -1e9, -1e9
            for stroke in variant:
                pts = []
                for nx, ny in stroke:
                    px = curr_x + nx * rw + tilt * (ny - 0.5) * box_h
                    py = y_top + ny * box_h
                    pts.append((px, py))
                    min_x, min_y = min(min_x, px), min(min_y, py)
                    max_x, max_y = max(max_x, px), max(max_y, py)
                draw_3d_chiseled_stroke(draw, pts, base_dark, stroke_w=lw)

            pad = 8
            bx1, by1 = max(0, min_x - pad), max(0, min_y - pad)
            bx2, by2 = min(img_size, max_x + pad), min(img_size, max_y + pad)
            cx, cy = (bx1 + bx2) / (2 * img_size), (by1 + by2) / (2 * img_size)
            bw, bh = (bx2 - bx1) / img_size, (by2 - by1) / img_size
            labels.append(f"{rune['id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            curr_x += rw + random.randint(20, 35)

    return img, labels

print("⏳ جاري توليد بيانات المرحلة 1 بدقة 1024x1024...")
for i in range(800):
    im, lbs = gen_s1_sample(i, sampler_s1)
    im.save(f'{d1}/images/train/s1_{i:04d}.jpg', quality=95)
    with open(f'{d1}/labels/train/s1_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

for i in range(150):
    im, lbs = gen_s1_sample(i, sampler_s1)
    im.save(f'{d1}/images/val/s1_val_{i:04d}.jpg', quality=95)
    with open(f'{d1}/labels/val/s1_val_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

# كتابة ملف dataset.yaml
y1 = f'{d1}/data.yaml'
with open(y1, 'w', encoding='utf-8') as f:
    yaml.dump({
        'path': str(Path(d1).resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES
    }, f)

print(f"✓ تم تجهيز 950 صورة بدقة 1024 للمرحلة 1.")

# ------------------------------------------------------------------------------
# إطلاق تدريب المرحلة الأولى
# ------------------------------------------------------------------------------
from ultralytics import YOLO

m1 = YOLO('yolov8m.pt')
m1.train(
    data=y1,
    epochs=35,
    imgsz=1024,          # دقة عالية لكشف الحروف الدقيقة
    batch=8,             # حجم الدفعة المتوافق مع T4 GPU عند 1024
    lr0=0.008,
    cos_lr=True,         # تدرج جيبي سلس لمعدل التعلم
    fliplr=0.0,          # الحفاظ التام على اتجاه الحروف
    flipud=0.0,
    degrees=6.0,
    project=str(DRIVE_PROJECT),
    name='curriculum_s1_1024',
    exist_ok=True,
    save=True
)

s1_best = DRIVE_PROJECT / 'curriculum_s1_1024' / 'weights' / 'best.pt'
s1_backup = DRIVE_PROJECT / 'viking_s1_best.pt'
if s1_best.exists():
    shutil.copy2(s1_best, s1_backup)
    print(f"🎉 تم حفظ أوزان المرحلة 1 في:
👉 {s1_backup}")


## 5. المرحلة الثانية (Stage 2: الأشرطة الكتابية المتصلة وفواصل الكلمات)
- **الهدف:** الانتقال إلى أشرطة كتابية كاملة تحتوي على فواصل الكلمات (`separator:`) مع تباين درجات الإضاءة وتآكل طفيف.
- **التدريب:** ينطلق من أوزان المرحلة الأولى `viking_s1_best.pt`.


In [ ]:
d2 = 'dataset_stage2'
os.makedirs(f'{d2}/images/train', exist_ok=True); os.makedirs(f'{d2}/labels/train', exist_ok=True)
os.makedirs(f'{d2}/images/val', exist_ok=True); os.makedirs(f'{d2}/labels/val', exist_ok=True)

sampler_s2 = BalancedRuneSampler(RUNES)
counts_s2 = Counter()

def gen_s2_sample(idx, sampler, img_size=1024):
    img = generate_photorealistic_rock(img_size, img_size)
    img = add_rock_cracks_and_veins(img)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(35, 60), random.randint(35, 60), random.randint(35, 60))
    lw = random.randint(4, 5)

    num_rows = random.choice([2, 3])
    for r in range(num_rows):
        num_runes = random.randint(7, 12)
        band_w = num_runes * 75
        x_start = max(50, (img_size - band_w) // 2 + random.randint(-40, 40))
        y_top = 100 + r * 280 + random.randint(-15, 15)
        box_h = 200

        # أشرطة تأطير منقوشة
        draw_3d_chiseled_stroke(draw, [(x_start - 30, y_top), (x_start + band_w + 30, y_top)], base_dark, stroke_w=3)
        draw_3d_chiseled_stroke(draw, [(x_start - 30, y_top + box_h), (x_start + band_w + 30, y_top + box_h)], base_dark, stroke_w=3)

        curr_x = x_start
        for item_idx in range(num_runes):
            # إدراج فواصل الكلمات الرونية (نقطتان أو علامة +)
            if item_idx > 0 and random.random() < 0.18:
                sep_id = 16
                counts_s2["separator"] += 1
                sep_w = random.randint(25, 35)
                # رسم نقطتين كفاصل
                draw_3d_chiseled_stroke(draw, [(curr_x + sep_w//2, y_top + box_h*0.35), (curr_x + sep_w//2, y_top + box_h*0.35 + 4)], base_dark, stroke_w=4)
                draw_3d_chiseled_stroke(draw, [(curr_x + sep_w//2, y_top + box_h*0.65), (curr_x + sep_w//2, y_top + box_h*0.65 + 4)], base_dark, stroke_w=4)

                bx1, by1 = curr_x, y_top + box_h * 0.25
                bx2, by2 = curr_x + sep_w, y_top + box_h * 0.75
                cx, cy = (bx1 + bx2) / (2 * img_size), (by1 + by2) / (2 * img_size)
                bw, bh = (bx2 - bx1) / img_size, (by2 - by1) / img_size
                labels.append(f"{sep_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                curr_x += sep_w + random.randint(15, 25)

            rune = sampler.next_rune()
            counts_s2[rune["name"]] += 1
            variant = random.choice(rune["variants"])
            rw = random.randint(55, 75)
            tilt = random.uniform(-0.05, 0.05)

            min_x, min_y, max_x, max_y = 1e9, 1e9, -1e9, -1e9
            for stroke in variant:
                pts = []
                for nx, ny in stroke:
                    px = curr_x + nx * rw + tilt * (ny - 0.5) * box_h
                    py = y_top + ny * box_h
                    pts.append((px, py))
                    min_x, min_y = min(min_x, px), min(min_y, py)
                    max_x, max_y = max(max_x, px), max(max_y, py)
                draw_3d_chiseled_stroke(draw, pts, base_dark, stroke_w=lw)

            pad = 6
            bx1, by1 = max(0, min_x - pad), max(0, min_y - pad)
            bx2, by2 = min(img_size, max_x + pad), min(img_size, max_y + pad)
            cx, cy = (bx1 + bx2) / (2 * img_size), (by1 + by2) / (2 * img_size)
            bw, bh = (bx2 - bx1) / img_size, (by2 - by1) / img_size
            labels.append(f"{rune['id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            curr_x += rw + random.randint(18, 28)

    return img, labels

print("⏳ جاري توليد بيانات المرحلة 2...")
for i in range(800):
    im, lbs = gen_s2_sample(i, sampler_s2)
    im.save(f'{d2}/images/train/s2_{i:04d}.jpg', quality=95)
    with open(f'{d2}/labels/train/s2_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

for i in range(150):
    im, lbs = gen_s2_sample(i, sampler_s2)
    im.save(f'{d2}/images/val/s2_val_{i:04d}.jpg', quality=95)
    with open(f'{d2}/labels/val/s2_val_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

y2 = f'{d2}/data.yaml'
with open(y2, 'w', encoding='utf-8') as f:
    yaml.dump({
        'path': str(Path(d2).resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES
    }, f)

# بدء تدريب المرحلة 2
s1_weights = str(s1_backup) if s1_backup.exists() else 'yolov8m.pt'
print(f"🚀 بدء تدريب المرحلة 2 انطلاقاً من: {s1_weights}")

m2 = YOLO(s1_weights)
m2.train(
    data=y2,
    epochs=40,
    imgsz=1024,
    batch=8,
    lr0=0.004,
    cos_lr=True,
    fliplr=0.0,
    flipud=0.0,
    scale=0.20,
    mosaic=0.4,
    close_mosaic=10,     # إيقاف التشويه في آخر 10 حقب لتثبيت الأشكال
    project=str(DRIVE_PROJECT),
    name='curriculum_s2_1024',
    exist_ok=True,
    save=True
)

s2_best = DRIVE_PROJECT / 'curriculum_s2_1024' / 'weights' / 'best.pt'
s2_backup = DRIVE_PROJECT / 'viking_s2_best.pt'
if s2_best.exists():
    shutil.copy2(s2_best, s2_backup)
    print(f"🎉 تم حفظ أوزان المرحلة 2 بنجاح:
👉 {s2_backup}")


## 6. المرحلة الثالثة (Stage 3: الكثافة العالية، التآكل الأثري، والدمج الهجين)
- **الهدف:** محاكاة التآكل الصخري الشديد وطبقات الطحالب (Patina/Lichen)، وتوليد نقوش حجرية كاملة.
- **التغذية الهجينة (Sim-to-Real):** إذا كان لديك ملف `viking_finetune_dataset.zip` مرفوعاً، سيتم دمج عينات منه لتعزيز التعرف الواقعي مباشرة!


In [ ]:
d3 = 'dataset_stage3'
os.makedirs(f'{d3}/images/train', exist_ok=True); os.makedirs(f'{d3}/labels/train', exist_ok=True)
os.makedirs(f'{d3}/images/val', exist_ok=True); os.makedirs(f'{d3}/labels/val', exist_ok=True)

sampler_s3 = BalancedRuneSampler(RUNES)
counts_s3 = Counter()

def apply_weathering_patina(img):
    w, h = img.size
    overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    o_draw = ImageDraw.Draw(overlay)
    # بقع رطوبة وطحالب خضراء/رمادية
    for _ in range(random.randint(2, 5)):
        lx, ly, lr = random.randint(50, w-50), random.randint(50, h-50), random.randint(60, 180)
        col = random.choice([(70, 95, 65, 35), (135, 125, 90, 40), (45, 50, 45, 30)])
        o_draw.ellipse([lx-lr, ly-lr, lx+lr, ly+lr], fill=col)
    overlay = overlay.filter(ImageFilter.GaussianBlur(18))
    img.paste(overlay, (0, 0), overlay)
    return img

def gen_s3_sample(idx, sampler, img_size=1024):
    img = generate_photorealistic_rock(img_size, img_size)
    img = add_rock_cracks_and_veins(img)
    draw = ImageDraw.Draw(img)
    labels = []
    base_dark = (random.randint(30, 55), random.randint(30, 55), random.randint(30, 55))
    lw = random.randint(3, 5)

    num_rows = random.choice([2, 3, 4])
    for r in range(num_rows):
        num_runes = random.randint(10, 16)
        band_w = num_runes * 58
        x_start = max(40, (img_size - band_w) // 2 + random.randint(-30, 30))
        y_top = 80 + r * 220 + random.randint(-10, 10)
        box_h = 160

        draw_3d_chiseled_stroke(draw, [(x_start - 25, y_top), (x_start + band_w + 25, y_top)], base_dark, stroke_w=2)
        draw_3d_chiseled_stroke(draw, [(x_start - 25, y_top + box_h), (x_start + band_w + 25, y_top + box_h)], base_dark, stroke_w=2)

        curr_x = x_start
        for item_idx in range(num_runes):
            if item_idx > 0 and random.random() < 0.20:
                counts_s3["separator"] += 1
                sep_w = random.randint(20, 28)
                draw_3d_chiseled_stroke(draw, [(curr_x + sep_w//2, y_top + box_h*0.35), (curr_x + sep_w//2, y_top + box_h*0.35 + 3)], base_dark, stroke_w=3)
                draw_3d_chiseled_stroke(draw, [(curr_x + sep_w//2, y_top + box_h*0.65), (curr_x + sep_w//2, y_top + box_h*0.65 + 3)], base_dark, stroke_w=3)
                bx1, by1 = curr_x, y_top + box_h * 0.25
                bx2, by2 = curr_x + sep_w, y_top + box_h * 0.75
                cx, cy = (bx1 + bx2) / (2 * img_size), (by1 + by2) / (2 * img_size)
                bw, bh = (bx2 - bx1) / img_size, (by2 - by1) / img_size
                labels.append(f"16 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                curr_x += sep_w + random.randint(10, 18)

            rune = sampler.next_rune()
            counts_s3[rune["name"]] += 1
            variant = random.choice(rune["variants"])
            rw = random.randint(42, 58)
            tilt = random.uniform(-0.06, 0.06)

            min_x, min_y, max_x, max_y = 1e9, 1e9, -1e9, -1e9
            for stroke in variant:
                pts = []
                for nx, ny in stroke:
                    px = curr_x + nx * rw + tilt * (ny - 0.5) * box_h
                    py = y_top + ny * box_h
                    pts.append((px, py))
                    min_x, min_y = min(min_x, px), min(min_y, py)
                    max_x, max_y = max(max_x, px), max(max_y, py)
                draw_3d_chiseled_stroke(draw, pts, base_dark, stroke_w=lw)

            pad = 5
            bx1, by1 = max(0, min_x - pad), max(0, min_y - pad)
            bx2, by2 = min(img_size, max_x + pad), min(img_size, max_y + pad)
            cx, cy = (bx1 + bx2) / (2 * img_size), (by1 + by2) / (2 * img_size)
            bw, bh = (bx2 - bx1) / img_size, (by2 - by1) / img_size
            labels.append(f"{rune['id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            curr_x += rw + random.randint(12, 22)

    img = apply_weathering_patina(img)
    return img, labels

print("⏳ جاري توليد بيانات المرحلة 3 (1000 صورة تدريب)...")
for i in range(1000):
    im, lbs = gen_s3_sample(i, sampler_s3)
    im.save(f'{d3}/images/train/s3_{i:04d}.jpg', quality=95)
    with open(f'{d3}/labels/train/s3_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

for i in range(200):
    im, lbs = gen_s3_sample(i, sampler_s3)
    im.save(f'{d3}/images/val/s3_val_{i:04d}.jpg', quality=95)
    with open(f'{d3}/labels/val/s3_val_{i:04d}.txt', 'w') as f:
        f.write('\n'.join(lbs))

# دمج صور حقيقية إذا كان ملف viking_finetune_dataset.zip متوفراً
real_zip = Path("viking_finetune_dataset.zip")
if not real_zip.exists() and (DRIVE_PROJECT / "viking_finetune_dataset.zip").exists():
    real_zip = DRIVE_PROJECT / "viking_finetune_dataset.zip"

if real_zip.exists():
    print("🌟 تم العثور على مجموعة الصور الحقيقية! جاري دمجها في التدريب لتقليص فجوة النطاق...")
    with zipfile.ZipFile(real_zip, 'r') as zf:
        zf.extractall("temp_real")
    real_imgs = list(Path("temp_real").rglob("*.jpg")) + list(Path("temp_real").rglob("*.png"))
    for idx, r_img in enumerate(real_imgs):
        lbl_file = r_img.with_suffix(".txt")
        if lbl_file.exists():
            shutil.copy2(r_img, f'{d3}/images/train/real_{idx:03d}.jpg')
            shutil.copy2(lbl_file, f'{d3}/labels/train/real_{idx:03d}.txt')
    print(f"✓ تم دمج {len(real_imgs)} صورة حقيقية بنجاح في بيانات المرحلة 3!")

y3 = f'{d3}/data.yaml'
with open(y3, 'w', encoding='utf-8') as f:
    yaml.dump({
        'path': str(Path(d3).resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES
    }, f)

# إطلاق التدريب النهائي
s2_weights = str(s2_backup) if s2_backup.exists() else 'yolov8m.pt'
print(f"👑 انطلاق تدريب الموديل الخبير الأخير من: {s2_weights}")

m3 = YOLO(s2_weights)
m3.train(
    data=y3,
    epochs=50,
    imgsz=1024,
    batch=8,
    lr0=0.002,           # معدل تعلم مصغر ومتقارب للحفاظ على الخصائص
    cos_lr=True,
    fliplr=0.0,
    flipud=0.0,
    degrees=6.0,
    scale=0.25,
    mosaic=0.3,
    close_mosaic=15,     # إيقاف الموازييك في آخر 15 حقبة لاستقرار البوكسات
    project=str(DRIVE_PROJECT),
    name='master_viking_stage3_1024',
    exist_ok=True,
    save=True
)

master_best = DRIVE_PROJECT / 'master_viking_stage3_1024' / 'weights' / 'best.pt'
final_master_model = DRIVE_PROJECT / 'viking_master_epigraphy_v2_best.pt'
if master_best.exists():
    shutil.copy2(master_best, final_master_model)
    print("=" * 75)
    print(f"🎉🎉🎉 تم إنتاج الموديل الخبير عالي الدقة بنجاح وحفظه في Google Drive:")
    print(f"👉 {final_master_model}")
    print("=" * 75)


## 7. التقييم النهائي وتنزيل الموديل (Evaluation & Download)
عرض مقاييس الدقة (Precision, Recall, mAP50) وحفظ وتنزيل الموديل مباشرة إلى جهاز الكمبيوتر.


In [ ]:
# تقييم الموديل النهائي
eval_model = YOLO(str(final_master_model))
metrics = eval_model.val(data=y3, imgsz=1024)

print("=" * 60)
print(f"  Precision (P)  : {metrics.box.mp:.4f} ({metrics.box.mp * 100:.2f}%)")
print(f"  Recall (R)     : {metrics.box.mr:.4f} ({metrics.box.mr * 100:.2f}%)")
print(f"  mAP@50         : {metrics.box.map50:.4f} ({metrics.box.map50 * 100:.2f}%)")
print(f"  mAP@50-95      : {metrics.box.map:.4f} ({metrics.box.map * 100:.2f}%)")
print("=" * 60)

# تنزيل الموديل مباشرة إلى المتصفح
from google.colab import files
if final_master_model.exists():
    print(f"⬇️ جاري بدء تنزيل الموديل الخبير: {final_master_model.name}")
    files.download(str(final_master_model))
